# Gold Layer: Arbitrage Opportunities Fact Notebook
Evaluates current ad listing prices against Gold market baselines to detect undervalued deals, retrieving `specs_summary` from `DimProduct` and calculating potential profit margins and opportunity scores.

## 1. Setup and Imports
Configure project root path and import dependencies alongside Silver and Gold layer model namespaces.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is available in system path
project_root = str(Path.cwd().parents[1])
if project_root not in sys.path:
    sys.path.append(project_root)

import polars as pl
from sqlalchemy import insert, select
from sqlalchemy.orm import Session

# Database configuration and model layer namespaces
from app.config import db_engine
from app.models import gold, silver

## 2. Load Listings, Product Dimensions, and Market Baselines
Fetch active clean ads from `silver.SilverCleanAd`, product spec summaries from `gold.DimProduct`, and benchmark pricing medians from `gold.FactMarketBaseline`.

In [ ]:
with db_engine.connect() as connection:
    # Query active clean ads from Silver layer
    df_silver_raw = pl.read_database(
        select(
            silver.SilverCleanAd.ad_id,
            silver.SilverCleanAd.title,
            silver.SilverCleanAd.url,
            silver.SilverCleanAd.first_image_src,
            silver.SilverCleanAd.item_condition,
            silver.SilverCleanAd.price.label('current_price'),
            silver.SilverCleanAd.baseline_id,
            silver.SilverCleanAd.item_condition_indicator,
            silver.SilverCleanAd.urgent_sale.label('is_urgent_sale')
        ),
        connection=connection
    )

    # Query specs_summary from Gold DimProduct dimension
    df_dim_products = pl.read_database(
        select(
            gold.DimProduct.baseline_id,
            gold.DimProduct.specs_summary
        ),
        connection=connection
    )

    # Query baseline benchmark medians from Gold FactMarketBaseline
    df_gold_baselines = pl.read_database(
        select(
            gold.FactMarketBaseline.baseline_id,
            gold.FactMarketBaseline.median_price.label('market_median_price')
        ),
        connection=connection
    )

## 3. Evaluate Arbitrage Deals and Calculate Metrics
Join listings with `DimProduct` (to retrieve `specs_summary`) and `FactMarketBaseline` (for `market_median_price`), filter for deals priced below baseline median, and compute profit, margin %, and opportunity scores.

In [ ]:
df_arbitrage_opportunities = (
    df_silver_raw
    # Join clean ads with product specs dimension
    .join(df_dim_products, on='baseline_id', how='inner')
    # Join clean ads with baseline benchmark pricing
    .join(df_gold_baselines, on='baseline_id', how='inner')
    # Filter deals priced below baseline median
    .filter(pl.col('current_price') < pl.col('market_median_price'))
    # Compute profit and margin metrics
    .with_columns(
        (pl.col('market_median_price') - pl.col('current_price')).round(2).alias('potential_profit'),
        (((pl.col('market_median_price') - pl.col('current_price')) / pl.col('current_price') * 100)).round(2).alias('profit_margin_pct')
    )
    # Compute opportunity score (0 to 100)
    .with_columns(
        (
            pl.col('profit_margin_pct').clip(0, 100) * 0.7
            + pl.when(pl.col('is_urgent_sale')).then(20).otherwise(0)
            + pl.when(pl.col('item_condition_indicator') == 1).then(10)
              .when(pl.col('item_condition_indicator') == 2).then(7)
              .when(pl.col('item_condition_indicator') == 3).then(5)
              .otherwise(0)
        ).clip(0, 100).round(1).alias('opportunity_score')
    )
    .drop('item_condition_indicator')
)

if not df_arbitrage_opportunities.is_empty():
    with Session(db_engine) as session:
        session.execute(
            insert(gold.FactArbitrageOpportunity), df_arbitrage_opportunities.to_dicts()
        )
        session.commit()
        print(f"Successfully detected and saved {len(df_arbitrage_opportunities)} arbitrage opportunities.")
else:
    print("No arbitrage opportunities found.")